**Avvio locale e Colab:** preparare la cartella completa seguendo il [README del laboratorio](README.md). Conservare gli identificativi delle celle e salvare gli output nel proprio runtime.

# [C01] Alignment & Calibration Studio — Computer Vision 2026

**Lezione 6 · Laurea Magistrale in Ingegneria Informatica, secondo anno**  
**Durata:** 3 ore · **Modalità:** coppie, confronto finale collettivo

## Domanda guida

> Quando un allineamento è credibile e quando una calibrazione è davvero informativa?

Pipeline osservabile:

**corrispondenze → stima di $H$ → consenso → warping → calibrazione → distorsione → posa**.

I dati principali sono sintetici e controllati: conosciamo la verità geometrica e possiamo distinguere un'immagine visivamente plausibile da una stima quantitativamente corretta.

Nei materiali del corso si usa il **punto decimale** (0.5) e la **virgola seguita da uno spazio** fra componenti di vettori e argomenti, per esempio (1.2, 1, −0.3). I conteggi interi non usano separatori delle migliaia.

## [C02] Percorso e consegne

Il corso è **Computer Vision**. Gli identificativi C01–C30 contano tutte le celle dall'alto (testo e codice), non i contatori variabili `In [ ]` dell'esecuzione. Usare gli identificativi Cxx nei commenti delle prove.

| Esperimento | Celle di codice | Evidenza richiesta |
| --- | --- | --- |
| Lab 0: protocollo e ambiente | C04 | versioni, seed, previsione |
| Lab 1: warping, canvas, interpolazione | C06, C08 | rettifica e confronto dei bordi |
| Lab 2: tutti i match / RANSAC | C12 | errore su punti di test e maschera |
| Lab 3: sweep outlier/soglia | C15, C16 | heatmap e regione stabile |
| Lab 4: calibrazione multi-vista | C19, C20 | K, distorsione, RMS e copertura |
| Lab 4B: distorsione e correzione | C23 | punti corretti e limiti del test |
| Lab 5: PnP robusto | C26 | posa, errori e inlier |
| discussione conclusiva | C28–C29 (testo) | diagnosi in cinque righe |

**Ordine da kernel nuovo:** C04 → C06 → C08 → C12 → C15 → C16 → C19 → C20 → C23 → C26. Non occorre eseguire le celle di testo. C15 richiede che C12 abbia già definito funzioni, H vera e griglia di test; C16 richiede i risultati di C15. C20 e C23 richiedono C19. C26 richiede C04 e C19. Se si riavvia il kernel, gli output visibili non ripristinano le variabili: rieseguire le dipendenze.

**Materiali da conservare:** notebook eseguito, tre figure scelte, due tabelle e diagnosi finale. Il solo valore RMS non è una validazione. C10 descrive l'uso di una fotografia propria, da preparare autonomamente; il percorso comune usa i dati sintetici inclusi.

## [C03] Lab 0 — Ambiente riproducibile

In Google Colab sono normalmente disponibili NumPy, Pandas, Matplotlib e OpenCV. In locale, se un import fallisce, installare i pacchetti elencati in `requirements_lezione06.txt` e riavviare il kernel.

In [ ]:
# C04 — Ambiente e seed
import sys
import time

import cv2 as cv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 20261001
rng = np.random.default_rng(SEED)
np.random.seed(SEED)
cv.setRNGSeed(SEED)
cv.setNumThreads(1)
plt.rcParams.update({
    "figure.dpi": 105, "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 12, "axes.labelsize": 10,
})

print("Python:", sys.version.split()[0])
print("OpenCV:", cv.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
assert int(cv.__version__.split(".")[0]) >= 4, "È richiesto OpenCV 4.x"

## [C05] Lab 1 — Warping: una trasformazione, due griglie

Una omografia trasforma coordinate. Un'immagine richiede anche una griglia di destinazione, una regola per i pixel fuori campo e un interpolatore.

`warpPerspective` riceve una trasformazione sorgente→destinazione, ma per riempire ogni pixel di output esegue concettualmente un **campionamento inverso**. Questo evita i buchi tipici del forward warping.

In [ ]:
# C06 — Documento e vista prospettica
CV_INK = (24, 33, 47)
CV_CYAN = (64, 197, 185)
CV_CORAL = (240, 80, 84)
CV_GOLD = (246, 192, 62)

def rgb(bgr):
    return cv.cvtColor(bgr, cv.COLOR_BGR2RGB)

def make_document(width=900, height=560):
    image = np.full((height, width, 3), (238, 234, 224), np.uint8)
    cv.rectangle(image, (0, 0), (width - 1, height - 1), CV_INK, 18)
    cv.putText(image, "COMPUTER VISION", (58, 88), cv.FONT_HERSHEY_DUPLEX,
               1.65, CV_INK, 4, cv.LINE_AA)
    cv.putText(image, "ALIGNMENT / CALIBRATION", (60, 145), cv.FONT_HERSHEY_SIMPLEX,
               0.92, CV_CORAL, 3, cv.LINE_AA)
    for row in range(5):
        for col in range(8):
            color = CV_GOLD if (row + col) % 2 else (45, 58, 74)
            x0, y0 = 65 + col * 58, 205 + row * 58
            cv.rectangle(image, (x0, y0), (x0 + 52, y0 + 52), color, -1)
    cv.circle(image, (700, 335), 112, CV_CYAN, -1, cv.LINE_AA)
    cv.circle(image, (700, 335), 68, CV_INK, 8, cv.LINE_AA)
    cv.line(image, (638, 273), (762, 397), CV_CORAL, 15, cv.LINE_AA)
    cv.line(image, (638, 397), (762, 273), CV_CORAL, 15, cv.LINE_AA)
    return image

document = make_document()
h, w = document.shape[:2]
src_corners = np.float32([[0, 0], [w - 1, 0], [w - 1, h - 1], [0, h - 1]])
scene_corners = np.float32([[170, 135], [1035, 75], [1120, 650], [95, 585]])
H_doc_to_scene = cv.getPerspectiveTransform(src_corners, scene_corners)
scene = cv.warpPerspective(document, H_doc_to_scene, (1220, 720),
                           flags=cv.INTER_LINEAR, borderValue=(12, 22, 35))

fig, axes = plt.subplots(1, 2, figsize=(15, 5.2))
axes[0].imshow(rgb(document)); axes[0].set_title("Piano di riferimento")
axes[1].imshow(rgb(scene)); axes[1].set_title("Vista prospettica sintetica")
for ax in axes: ax.axis("off")
plt.tight_layout()

### [C07] Esperimento 1A — Rettifica e interpolazione

Prevedere quale metodo produrrà bordi a gradino e quale tenderà a smussare. La rettifica usa $H^{-1}$ perché vogliamo mappare la scena nel sistema del documento.

In [ ]:
# C08 — Rettifica e interpolatori
H_scene_to_doc = np.linalg.inv(H_doc_to_scene)
methods = {"nearest": cv.INTER_NEAREST, "linear": cv.INTER_LINEAR, "cubic": cv.INTER_CUBIC}
rectified = {
    name: cv.warpPerspective(scene, H_scene_to_doc, (w, h), flags=flag,
                             borderValue=(12, 22, 35))
    for name, flag in methods.items()
}

fig, axes = plt.subplots(2, 3, figsize=(15, 8.2))
for col, (name, image) in enumerate(rectified.items()):
    axes[0, col].imshow(rgb(image)); axes[0, col].set_title(name)
    axes[1, col].imshow(rgb(image[175:405, 520:830])); axes[1, col].set_title("crop ad alta frequenza")
for ax in axes.ravel(): ax.axis("off")
plt.tight_layout()

warp_df = pd.DataFrame([
    {"interpolazione": name,
     "RMSE": np.sqrt(np.mean((image.astype(np.float32) - document.astype(np.float32)) ** 2))}
    for name, image in rectified.items()
]).sort_values("RMSE")
display(warp_df.round(3))

### [C09] Interpretazione

Quale interpolatore scegliereste per una maschera di segmentazione? E per un'immagine fotografica?

> Scrivere qui una risposta breve, citando almeno una misura o una figura.

#### Esercizio 1 — Bilineare e doppio ricampionamento

In un'immagine in scala di grigi, con coordinate $(u, v)$, sono noti i quattro campioni:

| Campione | Intensità |
|---|---:|
| $I(10, 20)$ | 40 |
| $I(11, 20)$ | 100 |
| $I(10, 21)$ | 80 |
| $I(11, 21)$ | 140 |

1. Calcolare l'intensità in $(10.75, 20.20)$ mediante **interpolazione bilineare ideale**, prima di arrotondamenti o quantizzazione. Esplicitare i quattro pesi, verificarne la somma e confrontare il risultato con nearest neighbor.
2. Nelle celle C06 e C08 il documento viene trasformato e poi rettificato. Anche disponendo dell'omografia esatta e della sua inversa, perché l'RMSE fra immagine iniziale e finale può essere diverso da zero? Distinguere l'inversione della trasformazione geometrica dal recupero dei campioni dell'immagine.

> Svolgimento: riportare i passaggi e motivare le risposte.

### [C10] Esperimento 1B — Input reale opzionale

In Colab si può sostituire `scene` con una fotografia di un foglio o poster. Selezionare i quattro vertici nello stesso ordine e conservare anche le corrispondenze: l'immagine rettificata da sola non rende l'esperimento riproducibile.

## [C11] Lab 2 — DLT e RANSAC sotto contaminazione controllata

Generiamo corrispondenze da una omografia nota, aggiungiamo rumore gaussiano e sostituiamo una frazione delle destinazioni con outlier uniformi. Valutiamo i modelli su punti mai usati nella stima.

**Metodo e metrica:** `findHomography(method=0)` usa DLT e rifinitura su tutti i match; non è un puro DLT lineare. I 100 match usati non sono 100 inlier verificati. L’errore della tabella è la mediana su 280 punti di test, confrontata con H vera: non è il residuo sui match di stima.

In [ ]:
# C12 — Stima su tutti i match e RANSAC
def project_points(H, points):
    points = np.asarray(points, np.float64).reshape(-1, 2)
    hom = np.c_[points, np.ones(len(points))]
    mapped = (H @ hom.T).T
    return mapped[:, :2] / mapped[:, 2:3]

H_gt = np.array([[1.08, 0.11, 85.0], [-0.06, 1.03, 55.0],
                 [1.8e-4, -1.2e-4, 1.0]], dtype=np.float64)

def contaminated_matches(outlier_fraction=0.35, noise_sigma=0.9, n=100, seed=SEED):
    local_rng = np.random.default_rng(seed)
    source = local_rng.uniform([40, 40], [920, 600], size=(n, 2)).astype(np.float32)
    target_clean = project_points(H_gt, source)
    target = target_clean + local_rng.normal(0, noise_sigma, target_clean.shape)
    is_inlier = np.ones(n, dtype=bool)
    count = int(round(outlier_fraction * n))
    idx = local_rng.choice(n, count, replace=False)
    target[idx] = local_rng.uniform([0, 0], [1100, 700], size=(count, 2))
    is_inlier[idx] = False
    return source, target.astype(np.float32), is_inlier

gx, gy = np.meshgrid(np.linspace(30, 930, 20), np.linspace(30, 610, 14))
test_points = np.c_[gx.ravel(), gy.ravel()]

def model_error(H):
    if H is None or not np.all(np.isfinite(H)):
        return np.inf
    predicted = project_points(H / H[2, 2], test_points)
    truth = project_points(H_gt, test_points)
    error = np.linalg.norm(predicted - truth, axis=1)
    return float(np.median(error)) if np.all(np.isfinite(error)) else np.inf

src, dst, truth_mask = contaminated_matches(outlier_fraction=0.35)
# method=0 usa un inizializzatore DLT e la rifinitura di OpenCV su tutti i match.
H_dlt, _ = cv.findHomography(src, dst, method=0)
H_ransac, estimated_mask = cv.findHomography(
    src, dst, method=cv.RANSAC, ransacReprojThreshold=3.0,
    maxIters=4000, confidence=0.999)
estimated_mask = estimated_mask.ravel().astype(bool)

comparison_df = pd.DataFrame([
    {"metodo": "DLT + rifinitura / tutti i match", "errore_test_mediano_px": model_error(H_dlt),
     "match_usati": len(src), "inlier_RANSAC": np.nan},
    {"metodo": "RANSAC + re-fit", "errore_test_mediano_px": model_error(H_ransac),
     "match_usati": len(src), "inlier_RANSAC": int(estimated_mask.sum())},
])
display(comparison_df.round(3))

fig, axes = plt.subplots(1, 2, figsize=(15, 5.4))
axes[0].scatter(dst[truth_mask, 0], dst[truth_mask, 1], s=24, c="#30D5E2", label="inlier veri")
axes[0].scatter(dst[~truth_mask, 0], dst[~truth_mask, 1], s=24, c="#FF6B72", label="outlier veri")
axes[0].set_title("Contaminazione nota"); axes[0].legend()
axes[1].scatter(dst[estimated_mask, 0], dst[estimated_mask, 1], s=24, c="#52D58B", label="consenso")
axes[1].scatter(dst[~estimated_mask, 0], dst[~estimated_mask, 1], s=24, c="#8FA5BA", label="rifiutati")
axes[1].set_title("Maschera RANSAC"); axes[1].legend()
for ax in axes:
    ax.set_aspect("equal", adjustable="box"); ax.invert_yaxis()
    ax.set_xlabel("u [px]"); ax.set_ylabel("v [px]")
plt.tight_layout()

### [C13] Interpretazione

Perché più corrispondenze non rendono automaticamente robusto il DLT?

> Scrivere qui una risposta breve, citando almeno una misura o una figura.

## [C14] Lab 3 — Sweep: quota di outlier × soglia RANSAC

Ripetiamo l'esperimento su più seed e costruiamo una mappa di stabilità. Prima di eseguire, prevedere dove RANSAC cederà e che cosa farà una soglia troppo stretta o troppo larga.

Ogni configurazione è ripetuta su 12 seed. La prima heatmap mostra la mediana dei 12 errori; la seconda la **frazione delle 12 prove** con errore mediano hold-out < 3 px. Una mediana bassa può coesistere con fallimenti: leggere entrambe le mappe.

In [ ]:
# C15 — Sweep ripetuto
outlier_grid = np.array([0.00, 0.15, 0.30, 0.45, 0.60, 0.75, 0.85])
threshold_grid = np.array([0.50, 1.00, 2.0, 3.0, 5.0, 8.0])
repetitions = 12

def run_trial(outlier_fraction, threshold, seed):
    src, dst, true_mask = contaminated_matches(outlier_fraction, noise_sigma=0.9, seed=seed)
    H, mask = cv.findHomography(src, dst, cv.RANSAC, float(threshold),
                                maxIters=800, confidence=0.99)
    if H is None or mask is None:
        return {"error": np.inf, "precision": 0.0, "recall": 0.0, "inlier_ratio": 0.0}
    estimated = mask.ravel().astype(bool)
    tp = np.sum(estimated & true_mask)
    fp = np.sum(estimated & ~true_mask)
    fn = np.sum(~estimated & true_mask)
    return {"error": model_error(H), "precision": tp / max(tp + fp, 1),
            "recall": tp / max(tp + fn, 1), "inlier_ratio": estimated.mean()}

rows = []
start = time.perf_counter()
for outlier_fraction in outlier_grid:
    for threshold in threshold_grid:
        for rep in range(repetitions):
            result = run_trial(outlier_fraction, threshold,
                               SEED + rep + int(1000 * outlier_fraction))
            rows.append({"outlier_fraction": outlier_fraction,
                         "threshold_px": threshold, "rep": rep, **result})
sweep_df = pd.DataFrame(rows)
# Un successo è una singola prova con mediana hold-out inferiore a 3 px.
sweep_df["successo_prova"] = sweep_df["error"] < 3.0
print(f"Sweep completato in {time.perf_counter() - start:.2f} s — {len(sweep_df)} stime")

summary = (sweep_df.groupby(["outlier_fraction", "threshold_px"], as_index=False)
           .agg(error_mediano_px=("error", "median"),
                precisione=("precision", "median"),
                recall=("recall", "median"),
                quota_inlier_stimata=("inlier_ratio", "median"),
                frazione_successi=("successo_prova", "mean")))
summary["mediana_sotto_3px"] = summary["error_mediano_px"] < 3.0
display(summary.round(3).head(12))

In [ ]:
# C16 — Heatmap e frequenza di successo
error_map = summary.pivot(index="outlier_fraction", columns="threshold_px",
                          values="error_mediano_px")
success_map = summary.pivot(index="outlier_fraction", columns="threshold_px",
                            values="frazione_successi")
display(error_map.round(2))

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
im0 = axes[0].imshow(np.clip(error_map.values, 0, 15), cmap="magma",
                     aspect="auto", vmin=0, vmax=15)
axes[0].set_title("Errore mediano su punti di test [px]")
im1 = axes[1].imshow(success_map.values.astype(float), cmap="viridis",
                     aspect="auto", vmin=0, vmax=1)
axes[1].set_title("Frazione di prove con errore < 3 px")
for ax in axes:
    ax.set_xticks(range(len(threshold_grid)), threshold_grid)
    ax.set_yticks(range(len(outlier_grid)), outlier_grid)
    ax.set_xlabel("soglia RANSAC τ [px]"); ax.set_ylabel("quota outlier")
fig.colorbar(im0, ax=axes[0], fraction=0.046)
fig.colorbar(im1, ax=axes[1], fraction=0.046, label="successi / 12 prove")
plt.tight_layout()

selected = summary[np.isclose(summary.outlier_fraction, 0.45)]
fig, ax = plt.subplots(figsize=(8.5, 4.8))
ax.plot(selected.threshold_px, selected.precisione, "o-", label="precisione")
ax.plot(selected.threshold_px, selected.recall, "o-", label="recall")
ax.set_ylim(0, 1.05); ax.set_xlabel("soglia RANSAC τ [px]")
ax.set_title("Trade-off con 45% di outlier"); ax.grid(alpha=.25); ax.legend()
plt.tight_layout()

### [C17] Interpretazione

Individuate una regione stabile. Quale soglia usereste con rumore di localizzazione circa 1 px?

> Scrivere qui una risposta breve, citando almeno una misura o una figura.

#### Esercizio 2 — Budget RANSAC e successo osservato

Per un'omografia il campione minimo contiene $s=4$ corrispondenze. Sia $w$ la **frazione di inlier**, non quella di outlier. Usare il modello approssimato con probabilità $w^4$ di estrarre un campione tutto inlier, estrazioni indipendenti e campioni tutti inlier non degeneri.

1. Per una probabilità desiderata $p=0.99$, calcolare il numero minimo di estrazioni
   $$N=\left\lceil\frac{\log(1-p)}{\log(1-w^4)}\right\rceil$$
   nei due casi $w=0.50$ e $w=0.25$.
2. C15 imposta `maxIters=800` e `confidence=0.99`. Nel caso $w=0.25$, calcolare $1-(1-w^4)^{800}$. Il limite di 800 estrazioni basta per raggiungere il 99% nel modello indicato?
3. Scegliere una configurazione della heatmap C16 e riportare quota di outlier, soglia in pixel e `frazione_successi`. Spiegare perché questa frequenza e il valore `confidence` non sono la stessa misura: quale evento è contato dalla formula e quale criterio definisce un successo in C15?

> Svolgimento: riportare i passaggi e motivare le risposte.

## [C18] Lab 4 — Calibrazione multi-vista: parametri e osservabilità

Simuliamo una scacchiera piana osservata da più pose. Gli intrinseci e la distorsione sono condivisi; ogni immagine ha la propria posa. Confrontiamo viste inclinate e distribuite con viste quasi frontali e centrali.

Nella riga ground truth l’RMS è NaN (non applicabile/non valutato): sono parametri generativi, non un fit ai corner rumorosi. Non interpretarlo come errore nullo.

In [ ]:
# C19 — Calibrazione buona e debole
IMAGE_SIZE = (1280, 720)
K_true = np.array([[900.0, 0.0, 640.0], [0.0, 880.0, 360.0], [0.0, 0.0, 1.0]])
dist_true = np.array([-0.22, 0.060, 0.0010, -0.0010, 0.0], dtype=np.float64)
cols, rows_board, square = 9, 6, 0.025
objp = np.zeros((rows_board * cols, 3), np.float32)
objp[:, :2] = np.mgrid[0:cols, 0:rows_board].T.reshape(-1, 2) * square

def make_calibration_views(n_views=14, quality="good", noise_sigma=0.25, seed=SEED):
    local_rng = np.random.default_rng(seed)
    object_points, image_points, poses = [], [], []
    attempts = 0
    while len(object_points) < n_views and attempts < 1000:
        attempts += 1
        if quality == "good":
            rvec = local_rng.uniform([-0.38, -0.38, -0.20],
                                     [0.38, 0.38, 0.20]).astype(np.float64)
            tvec = np.array([local_rng.uniform(-0.16, 0.10),
                             local_rng.uniform(-0.11, 0.08),
                             local_rng.uniform(0.58, 0.95)])
        else:
            rvec = local_rng.normal(0, [0.025, 0.025, 0.025]).astype(np.float64)
            tvec = np.array([local_rng.uniform(-0.025, 0.025),
                             local_rng.uniform(-0.018, 0.018),
                             local_rng.uniform(0.82, 0.98)])
        projected, _ = cv.projectPoints(objp, rvec, tvec, K_true, dist_true)
        projected = projected.reshape(-1, 2)
        visible = ((projected[:, 0] > 12) & (projected[:, 0] < IMAGE_SIZE[0] - 12) &
                   (projected[:, 1] > 12) & (projected[:, 1] < IMAGE_SIZE[1] - 12))
        if not visible.all():
            continue
        noisy = projected + local_rng.normal(0, noise_sigma, projected.shape)
        object_points.append(objp.copy())
        image_points.append(noisy.astype(np.float32))
        poses.append((rvec, tvec))
    if len(object_points) < n_views:
        raise RuntimeError("Impossibile generare tutte le viste")
    return object_points, image_points, poses

def calibrate_dataset(object_points, image_points):
    rms, K, dist, rvecs, tvecs = cv.calibrateCamera(
        object_points, image_points, IMAGE_SIZE, None, None)
    per_view = []
    for obj, observed, rvec, tvec in zip(object_points, image_points, rvecs, tvecs):
        predicted, _ = cv.projectPoints(obj, rvec, tvec, K, dist)
        residual = np.linalg.norm(predicted.reshape(-1, 2) - observed, axis=1)
        per_view.append(float(np.sqrt(np.mean(residual ** 2))))
    return {"rms": float(rms), "K": K, "dist": dist.ravel(),
            "rvecs": rvecs, "tvecs": tvecs, "per_view": np.array(per_view)}

good_obj, good_img, _ = make_calibration_views(quality="good")
weak_obj, weak_img, _ = make_calibration_views(quality="weak")
good_cal = calibrate_dataset(good_obj, good_img)
weak_cal = calibrate_dataset(weak_obj, weak_img)

def calibration_row(name, result):
    K, d = result["K"], result["dist"]
    return {"dataset": name, "RMS_px": result["rms"], "fx": K[0, 0],
            "fy": K[1, 1], "cx": K[0, 2], "cy": K[1, 2],
            "k1": d[0], "k2": d[1]}

# La verità riporta parametri generativi: non è un fit, RMS non valutato (NaN).
calibration_df = pd.DataFrame([
    {"dataset": "ground truth", "RMS_px": np.nan, "fx": K_true[0, 0],
     "fy": K_true[1, 1], "cx": K_true[0, 2], "cy": K_true[1, 2],
     "k1": dist_true[0], "k2": dist_true[1]},
    calibration_row("viste buone", good_cal),
    calibration_row("viste deboli", weak_cal),
])
display(calibration_df.round(4))

In [ ]:
# C20 — Copertura e residui per vista
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
for points in good_img: axes[0].plot(points[:, 0], points[:, 1], ".", alpha=.45)
axes[0].set_title("Viste inclinate e distribuite")
for points in weak_img: axes[1].plot(points[:, 0], points[:, 1], ".", alpha=.45)
axes[1].set_title("Viste quasi frontali")
axes[2].plot(good_cal["per_view"], "o-", label="viste buone")
axes[2].plot(weak_cal["per_view"], "o-", label="viste deboli")
axes[2].set_title("Residuo RMS per vista"); axes[2].set_xlabel("vista")
axes[2].set_ylabel("RMS [px]"); axes[2].legend()
for ax in axes[:2]:
    ax.set_xlim(0, IMAGE_SIZE[0]); ax.set_ylim(IMAGE_SIZE[1], 0)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("u [px]"); ax.set_ylabel("v [px]")
plt.tight_layout()

### [C21] Interpretazione

I due dataset possono avere RMS simili ma parametri diversi? Quale figura mostra l'informazione mancante?

> Scrivere qui una risposta breve, citando almeno una misura o una figura.

#### Esercizio 3 — Intrinseci dopo ridimensionamento e ritaglio

Usare gli intrinseci veri del caso sintetico:

$$K=\begin{bmatrix}900&0&640\\0&880&360\\0&0&1\end{bmatrix}.$$

In un caso analitico, l'immagine viene prima ridimensionata a metà lungo entrambi gli assi, poi ritagliata eliminando 20 pixel a sinistra e 10 in alto, **misurati nell'immagine già ridimensionata**. La convenzione delle coordinate è fissata da $u'=0.5u-20$, $v'=0.5v-10$, senza ulteriori offset del centro pixel.

1. Scrivere la matrice $A$ che applica questa trasformazione alle coordinate omogenee dell'immagine e calcolare $K'=AK$. Riportare $f'_x, f'_y, c'_x, c'_y$ in pixel.
2. A parità di camera e di modello di distorsione espresso in coordinate normalizzate, occorre modificare anche i coefficienti di distorsione? Motivare confrontando $(u'-c'_x)/f'_x$ con $(u-c_x)/f_x$ e analogamente per $v$. Spiegare che cosa accadrebbe usando la vecchia K per stimare la posa sulla nuova immagine.

> Svolgimento: riportare i passaggi e motivare le risposte.

### [C22] Esperimento 4B — Distorsione e correzione

Confrontiamo punti ideali, distorti e recuperati da `undistortPoints`. La distorsione radiale cresce con la distanza dal punto principale.

Questa prova usa K e distorsione **vere**, non i coefficienti stimati in C19: verifica l’andata/ritorno del modello noto, non la validità di una calibrazione reale. Con `P=K_true`, i punti corretti sono nel riferimento pixel di K vera.

In [ ]:
# C23 — Distorsione di punti e correzione
xx, yy = np.meshgrid(np.linspace(-0.60, 0.60, 15), np.linspace(-0.36, 0.36, 9))
plane3d = np.c_[xx.ravel(), yy.ravel(), np.ones(xx.size)].astype(np.float32)
zero = np.zeros(3)
ideal_px, _ = cv.projectPoints(plane3d, zero, zero, K_true, np.zeros(5))
distorted_px, _ = cv.projectPoints(plane3d, zero, zero, K_true, dist_true)
ideal_px = ideal_px.reshape(-1, 2)
distorted_px = distorted_px.reshape(-1, 2)
recovered_px = cv.undistortPoints(
    distorted_px.reshape(-1, 1, 2), K_true, dist_true, P=K_true).reshape(-1, 2)

distortion_error = np.linalg.norm(distorted_px - ideal_px, axis=1)
recovery_error = np.linalg.norm(recovered_px - ideal_px, axis=1)
print("Spostamento massimo: %.2f px" % distortion_error.max())
print("Errore massimo dopo correzione: %.6f px" % recovery_error.max())

fig, axes = plt.subplots(1, 2, figsize=(15, 5.3))
axes[0].quiver(ideal_px[:, 0], ideal_px[:, 1],
               distorted_px[:, 0] - ideal_px[:, 0],
               distorted_px[:, 1] - ideal_px[:, 1],
               distortion_error, angles="xy", scale_units="xy", scale=1, cmap="magma")
axes[0].scatter(ideal_px[:, 0], ideal_px[:, 1], s=8, c="#30D5E2")
axes[0].set_title("Ideale → distorto")
axes[1].scatter(ideal_px[:, 0], ideal_px[:, 1], s=25, facecolors="none",
                edgecolors="#30D5E2", label="ideale")
axes[1].scatter(recovered_px[:, 0], recovered_px[:, 1], s=8,
                c="#52D58B", label="recuperato")
axes[1].set_title("Dopo undistortPoints"); axes[1].legend()
for ax in axes:
    ax.set_xlim(0, IMAGE_SIZE[0]); ax.set_ylim(IMAGE_SIZE[1], 0)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("u [px]"); ax.set_ylabel("v [px]")
plt.tight_layout()

### [C24] Interpretazione

Perché una scacchiera sempre al centro è poco informativa per $k_1$ e $k_2$?

> Scrivere qui una risposta breve, citando almeno una misura o una figura.

## [C25] Lab 5 — PnP: da punti 3D–2D alla posa

Con $K$ e distorsione noti, PnP stima la trasformazione **oggetto/mondo → camera** $(R, t)$. Introduciamo un outlier e confrontiamo una stima diretta con `solvePnPRansac`.

La tabella confronta le proiezioni della posa stimata con `image2d`, cioè le proiezioni **ideali della posa vera**: la colonna `errore_proiezione_GT_mediano_px` non è un residuo rispetto alle osservazioni rumorose mostrate in figura. L’errore di rotazione e quello di traslazione richiedono anch’essi la verità sintetica. Non usare questi valori come misure disponibili su una camera reale priva di riferimento.

In [ ]:
# C26 — PnP diretto e robusto
object3d = np.float32([
    [-0.10, -0.08, 0.00], [0.10, -0.08, 0.00],
    [0.10, 0.08, 0.00], [-0.10, 0.08, 0.00],
    [-0.10, -0.08, 0.12], [0.10, -0.08, 0.12],
    [0.10, 0.08, 0.12], [-0.10, 0.08, 0.12],
])
rvec_true = np.array([0.24, -0.18, 0.08], dtype=np.float64)
tvec_true = np.array([0.045, -0.030, 0.88], dtype=np.float64)
image2d, _ = cv.projectPoints(object3d, rvec_true, tvec_true, K_true, dist_true)
image2d = image2d.reshape(-1, 2)
image2d_noisy = image2d + rng.normal(0, 0.65, image2d.shape)
image2d_noisy[6] += np.array([85.0, -70.0])

_, rvec_direct, tvec_direct = cv.solvePnP(
    object3d, image2d_noisy.astype(np.float32), K_true, dist_true,
    flags=cv.SOLVEPNP_EPNP)
_, rvec_robust, tvec_robust, pnp_inliers = cv.solvePnPRansac(
    object3d, image2d_noisy.astype(np.float32), K_true, dist_true,
    iterationsCount=1000, reprojectionError=3.0, confidence=0.999,
    flags=cv.SOLVEPNP_EPNP)

def rotation_error_deg(rvec_est, rvec_ref):
    R_est, _ = cv.Rodrigues(rvec_est)
    R_ref, _ = cv.Rodrigues(rvec_ref)
    value = np.clip((np.trace(R_est @ R_ref.T) - 1) / 2, -1, 1)
    return float(np.degrees(np.arccos(value)))

def pose_metrics(name, rvec, tvec):
    projected, _ = cv.projectPoints(object3d, rvec, tvec, K_true, dist_true)
    reproj = np.linalg.norm(projected.reshape(-1, 2) - image2d, axis=1)
    return {"metodo": name,
            "errore_rotazione_deg": rotation_error_deg(rvec, rvec_true),
            "errore_traslazione_m": float(np.linalg.norm(tvec.ravel() - tvec_true)),
            "errore_proiezione_GT_mediano_px": float(np.median(reproj))}

pnp_df = pd.DataFrame([
    pose_metrics("solvePnP diretto", rvec_direct, tvec_direct),
    pose_metrics("solvePnPRansac", rvec_robust, tvec_robust),
])
display(pnp_df.round(4))
print("Inlier PnP robusto:", pnp_inliers.ravel().tolist())

projected_direct, _ = cv.projectPoints(object3d, rvec_direct, tvec_direct, K_true, dist_true)
projected_robust, _ = cv.projectPoints(object3d, rvec_robust, tvec_robust, K_true, dist_true)
fig, ax = plt.subplots(figsize=(8.2, 5.3))
ax.scatter(image2d_noisy[:, 0], image2d_noisy[:, 1], s=55, c="#FF6B72", label="osservazioni")
ax.scatter(projected_direct[:, 0, 0], projected_direct[:, 0, 1],
           marker="x", s=65, c="#FFB45A", label="PnP diretto")
ax.scatter(projected_robust[:, 0, 0], projected_robust[:, 0, 1],
           marker="+", s=95, c="#52D58B", label="PnP robusto")
for idx, (u, v) in enumerate(image2d_noisy): ax.text(u + 4, v - 4, str(idx), fontsize=9)
ax.set_title("Osservazioni e riproiezioni"); ax.invert_yaxis()
ax.set_aspect("equal", adjustable="box"); ax.legend(); plt.tight_layout()

### [C27] Interpretazione

Che cosa restituiscono rvec e tvec? Perché il solo residuo non garantisce una posa accurata?

> Scrivere qui una risposta breve, citando almeno una misura o una figura.

#### Esercizio 4 — Invertire la posa e trovare il centro camera

Considerare il seguente **esempio analitico**, distinto dai risultati stimati in C26. La convenzione è $X_c=RX_o+t$: coordinate oggetto trasformate in coordinate camera, tutte in metri.

$$
R=\begin{bmatrix}0&-1&0\\1&0&0\\0&0&1\end{bmatrix}, \qquad
t=\begin{bmatrix}1\\2\\3\end{bmatrix}\ \mathrm{m}.
$$

1. Scrivere la trasformazione omogenea $T_{c\leftarrow o}$ e la sua inversa $T_{o\leftarrow c}$. Calcolare il centro della camera $C_o$ nel riferimento oggetto e verificarlo sostituendo in $RC_o+t=0$.
2. Trasformare il punto $X_o=(1, 0, 0)^T$ nel riferimento camera e poi riportarlo nel riferimento oggetto con l'inversa. Spiegare perché, in questo esempio, né $t$ né $-t$ rappresentano il centro della camera nel riferimento oggetto.

> Svolgimento: riportare i passaggi e motivare le risposte.

## [C28] Discussione conclusiva — Una diagnosi, non un numero

Completare in cinque righe:

1. modello stimato;
2. errore e unità;
3. quantità e copertura degli inlier;
4. degenerazione o instabilità osservata;
5. nuova acquisizione che migliorerebbe l'evidenza.

### [C29] Interpretazione

Scrivete la diagnosi come se doveste decidere se usare la calibrazione o l'omografia in un sistema reale.

> Scrivere qui una risposta breve, citando almeno una misura o una figura.

## [C30] Riferimenti essenziali

- A. Distante, C. Distante, *Visione computazionale: dall'energia all'immagine*: trasformazioni geometriche, interpolazione e calibrazione.
- R. Szeliski, *Computer Vision: Algorithms and Applications*, 2nd ed.: image alignment e camera calibration.
- R. Hartley, A. Zisserman, *Multiple View Geometry in Computer Vision*, 2nd ed.: DLT normalizzato, errore geometrico, stima robusta e posa.
- Documentazione OpenCV: `findHomography`, `warpPerspective`, `calibrateCamera`, `undistortPoints`, `solvePnP` e `solvePnPRansac`.